In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/vehicle_recalls.csv", dtype={"CAMPNO": str})
print(df.shape)
print(df[["battery_any", "fire_text", "battery_fire"]].dtypes)

(13721, 13)
battery_any     bool
fire_text       bool
battery_fire    bool
dtype: object


In [2]:
strata = {
    "A": df["battery_any"] & df["fire_text"],
    "B": df["battery_any"] & ~df["fire_text"],
    "C": ~df["battery_any"] & df["fire_text"],
    "D": ~df["battery_any"] & ~df["fire_text"],
}
sizes = {"A": 100, "B": 25, "C": 25, "D": 50}

for name, mask in strata.items():
    print(name, mask.sum())

A 347
B 235
C 2620
D 10519


In [3]:
parts = []
for name, mask in strata.items():
    s = df[mask].sample(n=sizes[name], random_state=42)
    s = s.assign(stratum=name)
    parts.append(s)

sample = pd.concat(parts).sample(frac=1, random_state=42).reset_index(drop=True)
sample["sample_id"] = range(1, len(sample) + 1)
print(sample["stratum"].value_counts())

stratum
A    100
D     50
C     25
B     25
Name: count, dtype: int64


In [4]:
to_label = sample[["sample_id", "DESC_DEFECT", "CONEQUENCE_DEFECT"]].assign(label="", note="")
to_label.to_csv("../data/labels/to_label.csv", index=False, encoding="utf-8-sig")

key = sample[["sample_id", "CAMPNO", "stratum", "battery_any", "fire_text", "battery_fire", "battery_type"]]
key.to_csv("../data/labels/sample_key.csv", index=False)

print(len(to_label), len(key))

200 200
